In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, element_at, trim, to_timestamp

In [3]:
spark = SparkSession.builder \
    .appName("BigDataProject_Ingestion") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [4]:
timetable_df = spark.read.option("header", "true").option("inferSchema", "true").csv("timetable_data.csv")
fares_df = spark.read.option("header", "true").option("inferSchema", "true").csv("fares_data.csv")
avl_df = spark.read.option("header", "true").option("inferSchema", "true").csv("avl_data.csv")
disruptions_df = spark.read.option("header", "true").option("inferSchema", "true").csv("disruptions_data.csv")

In [5]:
# Last colon-separated part of line_ref is the public route number
# e.g. "BLUS:PH1121126:200:5" -> "5", matching AVL/Disruptions format
timetable_df = timetable_df.withColumn("route_number", element_at(split(col("line_ref"), ":"), -1))
timetable_df = timetable_df.withColumn("departure_time", to_timestamp(col("departure_time"), "HH:mm:ss"))
timetable_df = timetable_df.dropDuplicates()

timetable_df.select("service_ref", "line_ref", "route_number", "departure_time").show(5, truncate=False)

+-------------+---------------------+------------+-------------------+
|service_ref  |line_ref             |route_number|departure_time     |
+-------------+---------------------+------------+-------------------+
|PH1121126:195|BLUS:PH1121126:195:18|18          |2026-07-21 06:52:00|
|PH1121126:195|BLUS:PH1121126:195:18|18          |2026-07-21 10:02:00|
|PH1121126:195|BLUS:PH1121126:195:18|18          |2026-07-21 11:52:00|
|PH1121126:195|BLUS:PH1121126:195:18|18          |2026-07-21 17:20:00|
|PH1121126:195|BLUS:PH1121126:195:18|18          |2026-07-21 17:40:00|
+-------------+---------------------+------------+-------------------+
only showing top 5 rows



In [6]:
fares_df = fares_df.withColumn("amount_gbp", col("amount_gbp").cast("double"))
fares_df = fares_df.dropDuplicates()

fares_df.select("operator_ref", "line_name", "amount_gbp").show(5, truncate=False)

+------------+----------------------------+----------+
|operator_ref|line_name                   |amount_gbp|
+------------+----------------------------+----------+
|BLUS        |Go South Coast 2605 Outbound|7.9       |
|BLUS        |Go South Coast 2607 Inbound |4.0       |
|BLUS        |Go South Coast 2607 Inbound |5.6       |
|BLUS        |Go South Coast 2607 Outbound|4.0       |
|BLUS        |Go South Coast 2627 Inbound |5.8       |
+------------+----------------------------+----------+
only showing top 5 rows



In [7]:
avl_df = avl_df.withColumnRenamed("line_ref", "route_number")
avl_df = avl_df.withColumn("latitude", col("latitude").cast("double"))
avl_df = avl_df.withColumn("longitude", col("longitude").cast("double"))
avl_df = avl_df.withColumn("recorded_at", to_timestamp(col("recorded_at")))
avl_df = avl_df.dropDuplicates()

avl_df.select("route_number", "operator_ref", "latitude", "longitude", "recorded_at").show(5, truncate=False)

+------------+------------+---------+---------+-------------------+
|route_number|operator_ref|latitude |longitude|recorded_at        |
+------------+------------+---------+---------+-------------------+
|18          |BLUS        |50.915414|-1.358222|2026-07-20 19:31:15|
|PRH         |BLUS        |50.935728|-1.461884|2026-07-20 19:31:16|
|8           |BLUS        |50.876635|-1.439586|2026-07-20 19:31:16|
|9           |BLUS        |50.88505 |-1.455481|2026-07-20 19:30:53|
|2           |BLUS        |50.965031|-1.323334|2026-07-20 19:31:10|
+------------+------------+---------+---------+-------------------+
only showing top 5 rows



In [8]:
disruptions_df = disruptions_df.withColumnRenamed("line_ref", "route_number")
disruptions_df = disruptions_df.withColumn("start_time", to_timestamp(col("start_time")))
disruptions_df = disruptions_df.withColumn("end_time", to_timestamp(col("end_time")))
disruptions_df = disruptions_df.dropDuplicates()

disruptions_df.select("disruption_id", "route_number", "operator_ref", "severity").show(5, truncate=False)

+-------------+------------+------------+--------+
|disruption_id|route_number|operator_ref|severity|
+-------------+------------+------------+--------+
|DIS-806578   |21          |WDBC        |medium  |
|DIS-546256   |8           |BLUS        |low     |
|DIS-240794   |203         |SVCT        |medium  |
|DIS-130455   |6           |SVCT        |low     |
|DIS-740822   |5           |DAMY        |low     |
+-------------+------------+------------+--------+
only showing top 5 rows



In [9]:
timetable_df = timetable_df.repartition(4)
fares_df = fares_df.repartition(4)
avl_df = avl_df.repartition(8)
disruptions_df = disruptions_df.repartition(4)

timetable_df.cache()
fares_df.cache()
avl_df.cache()
disruptions_df.cache()

DataFrame[disruption_id: string, operator_ref: string, route_number: string, start_time: timestamp, end_time: timestamp, duration_minutes: int, reason: string, severity: string, description: string]

In [10]:
print("Timetables:", timetable_df.count(), "rows,", timetable_df.rdd.getNumPartitions(), "partitions")
print("Fares:", fares_df.count(), "rows,", fares_df.rdd.getNumPartitions(), "partitions")
print("Location (AVL):", avl_df.count(), "rows,", avl_df.rdd.getNumPartitions(), "partitions")
print("Disruptions:", disruptions_df.count(), "rows,", disruptions_df.rdd.getNumPartitions(), "partitions")

total_records = timetable_df.count() + fares_df.count() + avl_df.count() + disruptions_df.count()
print("Total combined records:", total_records)

Timetables: 14391 rows, 4 partitions
Fares: 8980 rows, 4 partitions
Location (AVL): 128621 rows, 8 partitions
Disruptions: 3000 rows, 4 partitions
Total combined records: 154992


In [11]:
timetable_df.write.mode("overwrite").option("header", "true").csv("cleaned_timetable")
fares_df.write.mode("overwrite").option("header", "true").csv("cleaned_fares")
avl_df.write.mode("overwrite").option("header", "true").csv("cleaned_avl")
disruptions_df.write.mode("overwrite").option("header", "true").csv("cleaned_disruptions")

In [12]:
from pyspark.sql.functions import monotonically_increasing_id, unix_timestamp

avl_df = avl_df.withColumn("avl_id", monotonically_increasing_id())
avl_df = avl_df.withColumn("avl_time_of_day", unix_timestamp("recorded_at") % 86400)
timetable_df = timetable_df.withColumn("sched_time_of_day", unix_timestamp("departure_time") % 86400)

In [13]:
from pyspark.sql import Window
from pyspark.sql.functions import abs as spark_abs, row_number, col

# Left join keeps every AVL ping, even if a route has no timetable match
candidates_df = avl_df.join(
    timetable_df,
    on=["route_number", "operator_ref"],
    how="left"
)

candidates_df = candidates_df.withColumn(
    "time_diff", spark_abs(col("avl_time_of_day") - col("sched_time_of_day"))
)

# Keep only the closest scheduled trip per AVL ping
window = Window.partitionBy("avl_id").orderBy(col("time_diff").asc_nulls_last())

avl_timetable_df = candidates_df.withColumn("rank", row_number().over(window)) \
    .filter(col("rank") == 1) \
    .drop("rank")

print("AVL + nearest scheduled trip rows:", avl_timetable_df.count())

AVL + nearest scheduled trip rows: 128621


In [14]:
from pyspark.sql.functions import count as spark_count, avg as spark_avg

disruptions_summary_df = disruptions_df.groupBy("route_number", "operator_ref").agg(
    spark_count("disruption_id").alias("disruption_count"),
    spark_avg("duration_minutes").alias("avg_disruption_duration")
)

full_df = avl_timetable_df.join(
    disruptions_summary_df,
    on=["route_number", "operator_ref"],
    how="left"
)

full_df = full_df.fillna({"disruption_count": 0, "avg_disruption_duration": 0.0})

print("Joined + Disruptions rows:", full_df.count())

Joined + Disruptions rows: 128621


In [15]:
fares_summary_df = fares_df.groupBy("operator_ref").agg(
    spark_avg("amount_gbp").alias("avg_fare_gbp")
)

full_df = full_df.join(fares_summary_df, on="operator_ref", how="left")

print("Final combined rows:", full_df.count())

Final combined rows: 128621


In [16]:
full_df.write.mode("overwrite").option("header", "true").csv("combined_dataset")